In [ ]:
import os
import pandas as pd

def load_imdb_data(base_path):
    data = []
    for split in ["train", "test"]:
        for label in ["pos", "neg"]:
            folder = os.path.join(base_path, split, label)
            for fname in os.listdir(folder):
                with open(os.path.join(folder, fname), encoding="utf-8") as f:
                    text = f.read()
                data.append({
                    "text": text,
                    "label": 1 if label == "pos" else 0
                })
    return pd.DataFrame(data)

df = load_imdb_data("aclImdb")
print(df.head())


In [ ]:
df.to_csv("imdb_dataset.csv", index=False, encoding="utf-8")


Đã lưu thành công: imdb_clean.csv


In [2]:
import pandas as pd
df=pd.read_csv('/content/imdb_dataset.csv')

In [3]:
df.head()

,text,label
0,Bromwell High is a cartoon comedy. It ran at t...,1
1,Homelessness (or Houselessness as George Carli...,1
2,Brilliant over-acting by Lesley Ann Warren. Be...,1
3,This is easily the most underrated film inn th...,1
4,This is not the typical Mel Brooks film. It wa...,1


In [4]:
df.isnull().sum()

,0
text,0
label,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    50000 non-null  object
 1   label   50000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 781.4+ KB


In [6]:
df['label'].value_counts()

,count
label,
1,25000
0,25000


In [7]:
import re
import html
import unicodedata

def clean_text_bert(text):
    text = html.unescape(str(text))
    text = re.sub(r'<.*?>', ' ', text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'([,.!?])\1+', r'\1', text)  # thu gọn dấu câu lặp
    text = text.lower()  # nếu bạn dùng bert-base-uncased
    return text


In [8]:
bert_data=df['text'].apply(clean_text_bert)

In [9]:
bert_data.head()

,text
0,bromwell high is a cartoon comedy. it ran at t...
1,homelessness (or houselessness as george carli...
2,brilliant over-acting by lesley ann warren. be...
3,this is easily the most underrated film inn th...
4,this is not the typical mel brooks film. it wa...


In [10]:
print(bert_data[1])

homelessness (or houselessness as george carlin stated) has been an issue for years but never a plan to help those on the street that were once considered human who did everything from going to school, work, or vote for the matter. most people think of the homeless as just a lost cause while worrying about things such as racism, the war on iraq, pressuring kids to succeed, technology, the elections, inflation, or worrying if they'll be next to end up on the streets. but what if you were given a bet to live on the streets for a month without the luxuries you once had from a home, the entertainment sets, a bathroom, pictures on the wall, a computer, and everything you once treasure to see what it's like to be homeless? that is goddard bolt's lesson. mel brooks (who directs) who stars as bolt plays a rich man who has everything in the world until deciding to make a bet with a sissy rival (jeffery tambor) to see if he can live in the streets for thirty days without the luxuries; if bolt su

In [11]:
bert_df = pd.DataFrame({
    'text': bert_data,
    'label': df['label']
})


In [12]:
bert_df.shape

(50000, 2)

In [13]:
import torch
import time
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from tqdm import tqdm
from torch.optim import AdamW

In [14]:
class IMDBDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len
        )

        return {
            "input_ids": torch.tensor(encoding["input_ids"]),
            "attention_mask": torch.tensor(encoding["attention_mask"]),
            "labels": torch.tensor(self.labels[idx])
        }

In [15]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

device = "cuda" if torch.cuda.is_available() else "cpu"
scaler = torch.cuda.amp.GradScaler()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

/tmp/ipython-input-2551549575.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [16]:
device


'cuda'

In [17]:
tokenizer

DistilBertTokenizerFast(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [19]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accuracies = []
f1_scores = []
precisions = []
recalls = []
cms = []
train_times = []

num_epochs = 2
fold = 1

for train_idx, val_idx in skf.split(bert_df["text"], bert_df["label"]):

    print(f"\n========== FOLD {fold} ==========")
    print("→ Lấy dữ liệu Train/Validation...")

    X_train = bert_df["text"].iloc[train_idx].tolist()
    X_val   = bert_df["text"].iloc[val_idx].tolist()
    y_train = bert_df["label"].iloc[train_idx].tolist()
    y_val   = bert_df["label"].iloc[val_idx].tolist()

    print("  Train samples:", len(X_train))
    print("  Val samples:", len(X_val))

    # Dataset & DataLoader
    train_dataset = IMDBDataset(X_train, y_train, tokenizer)
    val_dataset   = IMDBDataset(X_val, y_val, tokenizer)

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)

    # Model
    model = DistilBertForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=2
    ).to(device)

    optimizer = AdamW(model.parameters(), lr=2e-5)
    total_steps = len(train_loader) * num_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )

    print("→ TRAINING...")
    start = time.time()
    model.train()

    for epoch in range(num_epochs):
        print(f"  Epoch {epoch+1}/{num_epochs}")
        for batch in tqdm(train_loader, desc="    Training"):
            optimizer.zero_grad()

            with torch.cuda.amp.autocast():
                outputs = model(
                    input_ids=batch["input_ids"].to(device),
                    attention_mask=batch["attention_mask"].to(device),
                    labels=batch["labels"].to(device)
                )
                loss = outputs.loss

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

    end = time.time()
    train_times.append(end - start)
    print(f"→ Train time: {end-start:.2f} sec")

    print("→ Đang EVALUATE...")
    model.eval()
    preds = []

    with torch.no_grad():
        for batch in val_loader:

            with torch.cuda.amp.autocast():
                outputs = model(
                    input_ids=batch["input_ids"].to(device),
                    attention_mask=batch["attention_mask"].to(device)
                )
                logits = outputs.logits

            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())

    acc = accuracy_score(y_val, preds)
    f1  = f1_score(y_val, preds, average='macro')
    prec = precision_score(y_val, preds, average='macro')
    rec  = recall_score(y_val, preds, average='macro')
    cm = confusion_matrix(y_val, preds)

    accuracies.append(acc)
    f1_scores.append(f1)
    precisions.append(prec)
    recalls.append(rec)
    cms.append(cm)

    print("\n===== METRICS =====")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1-macro: {f1:.4f}")
    print(f"Precision-macro: {prec:.4f}")
    print(f"Recall-macro: {rec:.4f}")

    print("\n===== CONFUSION MATRIX (FOLD) =====")
    print(cm)

    print("\n===== 5 SAMPLE PREDICTIONS =====")
    for i in range(min(5, len(X_val))):
        print(f"\n--- Sample {i+1} ---")
        print("TEXT:", X_val[i][:300], ("..." if len(X_val[i]) > 300 else ""))
        print("Thực tế:", y_val[i])
        print("Dự đoán:", preds[i])

    fold += 1
print("\n==============================")
print(f"Accuracy AVG: {np.mean(accuracies):.4f}")
print(f"Accuracy STD: {np.std(accuracies):.4f}")
print("------------------------------")
print(f"F1 AVG: {np.mean(f1_scores):.4f}")
print(f"F1 STD: {np.std(f1_scores):.4f}")
print("------------------------------")
print(f"Precision AVG: {np.mean(precisions):.4f}")
print(f"Recall AVG: {np.mean(recalls):.4f}")
print("------------------------------")
print(f"Train time AVG: {np.mean(train_times):.2f} sec")
print("==============================")



========== FOLD 1 ==========
→ Lấy dữ liệu Train/Validation...
  Train samples: 40000
  Val samples: 10000


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


→ TRAINING...
  Epoch 1/2


    Training:   0%|          | 0/2500 [00:00<?, ?it/s]/tmp/ipython-input-1834129106.py:56: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
    Training: 100%|██████████| 2500/2500 [04:27<00:00,  9.36it/s]


  Epoch 2/2


...

→ TRAINING...
  Epoch 1/2


    Training:   0%|          | 0/2500 [00:00<?, ?it/s]/tmp/ipython-input-1834129106.py:56: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
    Training: 100%|██████████| 2500/2500 [04:39<00:00,  8.94it/s]


  Epoch 2/2


    Training: 100%|██████████| 2500/2500 [04:40<00:00,  8.92it/s]
/tmp/ipython-input-1834129106.py:80: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


→ Train time: 559.70 sec
→ Đang EVALUATE...

===== METRICS =====
Accuracy: 0.9215
F1-macro: 0.9215
Precision-macro: 0.9216
Recall-macro: 0.9215

===== CONFUSION MATRIX (FOLD) =====
[[4561  439]
 [ 346 4654]]

===== 5 SAMPLE PREDICTIONS =====

--- Sample 1 ---
TEXT: homelessness (or houselessness as george carlin stated) has been an issue for years but never a plan to help those on the street that were once considered human who did everything from going to school, work, or vote for the matter. most people think of the homeless as just a lost cause while worryin ...
Thực tế: 1
Dự đoán: 1

--- Sample 2 ---
TEXT: when i first read armistead maupins story i was taken in by the human drama displayed by gabriel no one and those he cares about and loves. that being said, we have now been given the film version of an excellent story and are expected to see past the gloss of hollywood. writer armistead maupin and  ...
Thực tế: 1
Dự đoán: 1

--- Sample 3 ---
TEXT: the night listener held my atten

train mô hình với full dataset

In [20]:
device = "cuda" if torch.cuda.is_available() else "cpu"

MAX_LEN = 256
BATCH_SIZE = 16
LR = 2e-5
EPOCHS = 2

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
).to(device)

class FullDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        enc = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

texts = bert_df['text'].tolist()
labels = bert_df['label'].tolist()

dataset_full = FullDataset(texts, labels)
loader_full = DataLoader(dataset_full, batch_size=BATCH_SIZE, shuffle=True)

optimizer = AdamW(model.parameters(), lr=LR)
scaler = torch.cuda.amp.GradScaler()

model.train()

for epoch in range(EPOCHS):
    print(f"\n===== EPOCH {epoch+1}/{EPOCHS} ====")
    pbar = tqdm(loader_full)

    for batch in pbar:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.amp.autocast("cuda"):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        pbar.set_description(f"Loss {loss.item():.4f}")

print(">> TRAINING FULL-DATA DONE!")
# SAVE MODEL
model.save_pretrained("distilbert")
tokenizer.save_pretrained("distilbert")

print(">> Model saved to distilbert/")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2023774123.py:47: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()



===== EPOCH 1/2 ====


Loss 0.1313: 100%|██████████| 3125/3125 [06:35<00:00,  7.90it/s]



===== EPOCH 2/2 ====


Loss 0.2845: 100%|██████████| 3125/3125 [06:37<00:00,  7.87it/s]


>> TRAINING FULL-DATA DONE!
>> Model saved to distilbert/


inference với một vài sample

In [21]:
import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert")
model = DistilBertForSequenceClassification.from_pretrained("distilbert").to(device)
model.eval()


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [22]:
def predict_bert(text):
    enc = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=256,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        logits = model(**enc).logits
        pred = torch.argmax(logits, dim=1).item()

    return pred


In [23]:
sentences = [
    "The movie was absolutely amazing!",
    "I hate this film, it was terrible.",
    "The plot was ok but the ending felt weak."
]

print("\n===== BERT INFERENCE ON 3 SENTENCES =====")

for i, s in enumerate(sentences):
    pred = predict_bert(s)
    print(f"\nSentence {i+1}:")
    print("TEXT :", s)
    print("PRED :", pred)



===== BERT INFERENCE ON 3 SENTENCES =====

Sentence 1:
TEXT : The movie was absolutely amazing!
PRED : 1

Sentence 2:
TEXT : I hate this film, it was terrible.
PRED : 0

Sentence 3:
TEXT : The plot was ok but the ending felt weak.
PRED : 0
